# Track A — A1/A2 공식 모델 재현 + F0/F1/F2 민감도 비교 (track_a_01)

담당: 김태헌 | 입력: 태영님 전달 파일 7종 (track_a_train_A1/A2_v2.csv 등)

## 이 노트북의 목적

`전달 파일 7종 설명` 문서에 이미 확정된 모델링 방법론(STEP 17-30 튜닝 결과)을 하나의
깨끗한 노트북으로 재현한다. 새로 튜닝하지 않고 **문서에 명시된 확정 설정을 그대로 사용**한다.

- **A-2 (본론 모델)**: 진입 전 씬파일러 시절 정보 10개 피처. "기록이 전혀 없는 사람을
  대안정보만으로 평가할 수 있는가"라는 프로젝트 핵심 질문 자체
- **A-1 (비교 기준자)**: 진입 시점 정보 15개 피처. A-2의 가치를 재는 눈금(CB 완전정보
  ≈0.99 → A1 0.858 → A2 0.820, 모두 인구변수 포함 기준)

## 문서가 요청한 6가지 요구사항 반영

1. ID 기준 `StratifiedGroupKFold(5)`, 주 타겟 `TARGET_24M`, `TARGET_12M` 병행 보고
2. `GENDER`·`AGE_BAND`는 공식 모델 피처에서 제외(F2). 포함 버전(F0)·성별만 제외(F1)는 참고치로 병기
3. 불균형: 다운샘플 50:1 × 5앙상블 (SMOTE는 팀이 이미 실측으로 이득 없음을 확인함 — STEP 21R/21T)
4. 성능 보고: 부트스트랩 95% CI 병기, fold 표준편차 단독 표기 지양
5. WoE 인코딩(구간화 파일 사용하는 로지스틱 계열)은 이 노트북 범위 밖 — 별도 노트북(track_a_02)에서 진행
6. 확률이 아니라 순위·등급 용도로 해석 — 미보정 확률이라는 점을 결과 표에 명시

## ⚠️ 확인된 데이터 특성

- `ID`가 완전히 유일하지 않음(68,677행 중 68,367개 고유 ID, 310개 중복) → `StratifiedGroupKFold`가
  형식적 요구가 아니라 실제로 필요한 상황
- `DELTA_*_prev` 3개 컬럼이 ENTRY_YEAR=2020인 8,150건 전부 결측 — 진입 2년 전 데이터가 없어
  변화량 계산이 불가능하기 때문. 결측치는 0으로 채우고 별도 플래그 없이 처리(팀 확정 방식)


## 0. 환경 설정

In [ ]:
# %pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
# 로컬 환경: 터미널에서 아래 명령어로 한 번만 설치해두면 됨
# pip install lightgbm

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import json

RANDOM_STATE = 42
handoff_path = r'C:\Users\tehun\Desktop\multicamp\project\creditscore\personalCB'  # 필요시 수정

# 팀이 STEP 21T에서 확정한 트리 파라미터 (재탐색 불필요)
LGB_PARAMS = dict(
    max_depth=2, num_leaves=4, min_child_samples=20,
    n_estimators=200, learning_rate=0.05,
    objective='binary', verbose=-1,
)
DOWNSAMPLE_RATIO = 50   # 음성:양성 = 50:1
N_ENSEMBLE = 5          # 다운샘플 앙상블 모델 수


## 1. 데이터 로드 + 공식 피처 목록 정의

In [8]:
df_a1 = pd.read_csv(f'{handoff_path}/track_a_train_A1_v2.csv')
df_a2 = pd.read_csv(f'{handoff_path}/track_a_train_A2_v2.csv')

print(f"A1: {df_a1.shape}, A2: {df_a2.shape}")

# 문서에 명시된 공식 피처 목록 (그대로 사용, 여분 컬럼은 사용하지 않음)
FEATURES_A2 = [  # 공식 F2 모델 피처 10개
    'AL012G005_prev', 'AL012G011_prev', 'AL012G019_prev',
    'DELTA_AL012G005_prev', 'DELTA_AL012G011_prev', 'DELTA_AL012G019_prev',
    'U81301010_prev', 'U81305010_prev', 'U81306010_prev',
    'AS120G001_prev',
]

FEATURES_A1 = [  # 공식 15개 (공통 10 + A-1 전용 5)
    'AL012G005', 'AL012G011', 'AL012G019',
    'DELTA_AL012G005', 'DELTA_AL012G011', 'DELTA_AL012G019',
    'U81301010', 'U81305010', 'U81306010', 'AS120G001',
    'CAR_FLAG', 'NONBANK_RATIO',
    'U81201010_CONF_MISSING', 'U81301010_CONF_MISSING', 'U81302010_CONF_MISSING',
]

POP_COLS = ['GENDER', 'AGE_BAND']  # 인구변수 (F0/F1 참고치에만 사용, 공식 F2에서는 제외)

# ID 중복 확인 (StratifiedGroupKFold 필요성 검증)
for name, df in [('A1', df_a1), ('A2', df_a2)]:
    n_dup = df['ID'].duplicated().sum()
    print(f"{name}: 전체 {len(df)}행, 고유 ID {df['ID'].nunique()}개, 중복 {n_dup}개")

# 결측치 처리 (DELTA_* 계열, 팀 확정 방식: 0으로 채움)
for df in [df_a1, df_a2]:
    delta_cols = [c for c in df.columns if c.startswith('DELTA_')]
    df[delta_cols] = df[delta_cols].fillna(0)


A1: (68677, 22), A2: (68677, 23)
A1: 전체 68677행, 고유 ID 68367개, 중복 310개
A2: 전체 68677행, 고유 ID 68367개, 중복 310개


## 2. 다운샘플 앙상블 + StratifiedGroupKFold OOF 학습 함수

팀이 이미 확정한 방식(50:1 다운샘플 × 5모델 앙상블, LightGBM depth2/leaves4/min_child20)을
그대로 구현한다. Out-of-fold(OOF) 예측을 모아서 전체 성능을 계산 — 자기 문제로 채점하는
부풀림 없이 정직한 성능 추정치를 얻는 방식.

In [9]:
def downsample_ensemble_oof(X, y, groups, n_splits=5, ratio=DOWNSAMPLE_RATIO,
                              n_ensemble=N_ENSEMBLE, seed=RANDOM_STATE, params=None):
    params = params or LGB_PARAMS
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_proba = np.zeros(len(y))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, y, groups)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        pos_idx = y_tr[y_tr == 1].index
        neg_idx_all = y_tr[y_tr == 0].index
        n_neg_sample = min(len(neg_idx_all), len(pos_idx) * ratio)

        fold_pred = np.zeros(len(va_idx))
        for e in range(n_ensemble):
            rng = np.random.RandomState(seed + fold * 1000 + e)
            neg_sample = rng.choice(neg_idx_all, size=n_neg_sample, replace=False)
            train_idx = np.concatenate([pos_idx.values, neg_sample])

            Xe, ye = X_tr.loc[train_idx], y_tr.loc[train_idx]
            model = lgb.LGBMClassifier(**params, random_state=seed + e)
            model.fit(Xe, ye)
            fold_pred += model.predict_proba(X_va)[:, 1] / n_ensemble

        oof_proba[va_idx] = fold_pred
        fold_auc = roc_auc_score(y_va, fold_pred) if y_va.sum() > 0 else np.nan
        fold_aucs.append(fold_auc)
        print(f"  fold {fold+1}: 양성 {y_tr.sum()}건(train) / {y_va.sum()}건(valid), AUC={fold_auc:.4f}")

    overall_auc = roc_auc_score(y, oof_proba)
    return oof_proba, overall_auc, fold_aucs


## 3. 부트스트랩 95% CI 함수

fold 표준편차 대신, 사람(ID) 단위 복원추출로 신뢰구간을 산출한다(팀 확정 방식,
"fold ±표준편차 → 부트스트랩 95% CI"로 대체됨).

In [10]:
def bootstrap_ci_auc(y_true, y_proba, n_boot=2000, seed=RANDOM_STATE, ci=95):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_proba = np.asarray(y_proba)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        if yt.sum() == 0 or yt.sum() == n:
            continue
        aucs.append(roc_auc_score(yt, yp))
    lo, hi = (100 - ci) / 2, 100 - (100 - ci) / 2
    return float(np.mean(aucs)), (float(np.percentile(aucs, lo)), float(np.percentile(aucs, hi)))


## 4. A-2(본론 모델) 학습 — F0(인구포함) / F1(성별만 제외) / F2(공식, 인구 전부 제외)

공식 수치는 F2다. F0·F1은 "인구변수를 얼마나 배제해도 성능이 유지되는가"를 보여주는
민감도 비교용 참고치로만 병기한다(차별금지 모범규준 대응).

In [11]:
y24 = df_a2['TARGET_24M']
groups_a2 = df_a2['ID']

variants_a2 = {
    'F0 (인구 포함)':      FEATURES_A2 + POP_COLS,
    'F1 (성별만 제외)':     FEATURES_A2 + ['AGE_BAND'],
    'F2 (인구 전부 제외, 공식)': FEATURES_A2,
}

results_a2_24m = {}
oof_arrays_a2_24m = {}  # 8절(apply 대조)에서 재사용하기 위해 저장
for name, feat_cols in variants_a2.items():
    print(f"=== A2 — {name} (TARGET_24M) ===")
    X = df_a2[feat_cols]
    oof_proba, auc, fold_aucs = downsample_ensemble_oof(X, y24, groups_a2)
    ci_mean, ci_range = bootstrap_ci_auc(y24, oof_proba)
    results_a2_24m[name] = {
        'oof_auc': auc, 'bootstrap_mean': ci_mean, 'bootstrap_ci': ci_range,
        'n_features': len(feat_cols),
    }
    oof_arrays_a2_24m[name] = oof_proba
    print(f"  OOF AUROC: {auc:.4f}, 부트스트랩 95% CI: [{ci_range[0]:.4f}, {ci_range[1]:.4f}]\n")

pd.DataFrame(results_a2_24m).T


=== A2 — F0 (인구 포함) (TARGET_24M) ===
  fold 1: 양성 115건(train) / 29건(valid), AUC=0.8092
  fold 2: 양성 115건(train) / 29건(valid), AUC=0.8741
  fold 3: 양성 115건(train) / 29건(valid), AUC=0.7580
  fold 4: 양성 116건(train) / 28건(valid), AUC=0.8127
  fold 5: 양성 115건(train) / 29건(valid), AUC=0.8538
  OOF AUROC: 0.8190, 부트스트랩 95% CI: [0.7825, 0.8517]

=== A2 — F1 (성별만 제외) (TARGET_24M) ===
  fold 1: 양성 115건(train) / 29건(valid), AUC=0.7788
  fold 2: 양성 115건(train) / 29건(valid), AUC=0.8466
  fold 3: 양성 115건(train) / 29건(valid), AUC=0.7239
  fold 4: 양성 116건(train) / 28건(valid), AUC=0.8029
  fold 5: 양성 115건(train) / 29건(valid), AUC=0.8304
  OOF AUROC: 0.7933, 부트스트랩 95% CI: [0.7556, 0.8278]

=== A2 — F2 (인구 전부 제외, 공식) (TARGET_24M) ===
  fold 1: 양성 115건(train) / 29건(valid), AUC=0.6454
  fold 2: 양성 115건(train) / 29건(valid), AUC=0.6646
  fold 3: 양성 115건(train) / 29건(valid), AUC=0.6820
  fold 4: 양성 116건(train) / 28건(valid), AUC=0.7146
  fold 5: 양성 115건(train) / 29건(valid), AUC=0.7451
  OOF AUROC: 0.6896, 부트스트

,oof_auc,bootstrap_mean,bootstrap_ci,n_features
F0 (인구 포함),0.81897,0.818517,"(0.7824951516380582, 0.8517079871843363)",12
F1 (성별만 제외),0.793327,0.793143,"(0.7556202916778753, 0.8277507146414751)",11
"F2 (인구 전부 제외, 공식)",0.689598,0.690004,"(0.6464101487274135, 0.7311441197097875)",10


## 5. A-2(본론 모델) — TARGET_12M 병행 보고

양성 96건 기준. 24M과 기준선(양성률)이 다르므로 직접 비교하지 않고 따로 보고한다.

In [12]:
y12 = df_a2['TARGET_12M']

results_a2_12m = {}
for name, feat_cols in variants_a2.items():
    print(f"=== A2 — {name} (TARGET_12M) ===")
    X = df_a2[feat_cols]
    oof_proba, auc, fold_aucs = downsample_ensemble_oof(X, y12, groups_a2)
    ci_mean, ci_range = bootstrap_ci_auc(y12, oof_proba)
    results_a2_12m[name] = {
        'oof_auc': auc, 'bootstrap_mean': ci_mean, 'bootstrap_ci': ci_range,
        'n_features': len(feat_cols),
    }
    print(f"  OOF AUROC: {auc:.4f}, 부트스트랩 95% CI: [{ci_range[0]:.4f}, {ci_range[1]:.4f}]\n")

pd.DataFrame(results_a2_12m).T


=== A2 — F0 (인구 포함) (TARGET_12M) ===
  fold 1: 양성 76건(train) / 20건(valid), AUC=0.8358
  fold 2: 양성 77건(train) / 19건(valid), AUC=0.7027
  fold 3: 양성 77건(train) / 19건(valid), AUC=0.7878
  fold 4: 양성 77건(train) / 19건(valid), AUC=0.8430
  fold 5: 양성 77건(train) / 19건(valid), AUC=0.8780
  OOF AUROC: 0.8064, 부트스트랩 95% CI: [0.7597, 0.8492]

=== A2 — F1 (성별만 제외) (TARGET_12M) ===
  fold 1: 양성 76건(train) / 20건(valid), AUC=0.7911
  fold 2: 양성 77건(train) / 19건(valid), AUC=0.6397
  fold 3: 양성 77건(train) / 19건(valid), AUC=0.7447
  fold 4: 양성 77건(train) / 19건(valid), AUC=0.8186
  fold 5: 양성 77건(train) / 19건(valid), AUC=0.8761
  OOF AUROC: 0.7673, 부트스트랩 95% CI: [0.7159, 0.8150]

=== A2 — F2 (인구 전부 제외, 공식) (TARGET_12M) ===
  fold 1: 양성 76건(train) / 20건(valid), AUC=0.6730
  fold 2: 양성 77건(train) / 19건(valid), AUC=0.5733
  fold 3: 양성 77건(train) / 19건(valid), AUC=0.6601
  fold 4: 양성 77건(train) / 19건(valid), AUC=0.7013
  fold 5: 양성 77건(train) / 19건(valid), AUC=0.7823
  OOF AUROC: 0.6756, 부트스트랩 95% CI: [0.61

,oof_auc,bootstrap_mean,bootstrap_ci,n_features
F0 (인구 포함),0.806394,0.805952,"(0.759730818892083, 0.8492135213900734)",12
F1 (성별만 제외),0.767339,0.767191,"(0.7159226304503283, 0.8150373162993955)",11
"F2 (인구 전부 제외, 공식)",0.675569,0.676076,"(0.6195334463000182, 0.7311188307889689)",10


## 6. A-1(비교 기준자) 학습 — 인구 포함(F0)만

A-1은 A-2의 가치를 재는 눈금 역할이라 F0 하나만 계산한다(문서의 3단 비교 구도:
CB 완전정보 ≈0.99 → A1 → A2).

In [13]:
groups_a1 = df_a1['ID']
X_a1 = df_a1[FEATURES_A1 + POP_COLS]

print("=== A1 — F0 (인구 포함) (TARGET_24M) ===")
oof_a1_24, auc_a1_24, _ = downsample_ensemble_oof(X_a1, df_a1['TARGET_24M'], groups_a1)
ci_mean_a1, ci_range_a1 = bootstrap_ci_auc(df_a1['TARGET_24M'], oof_a1_24)
print(f"OOF AUROC: {auc_a1_24:.4f}, 부트스트랩 95% CI: [{ci_range_a1[0]:.4f}, {ci_range_a1[1]:.4f}]")


=== A1 — F0 (인구 포함) (TARGET_24M) ===
  fold 1: 양성 115건(train) / 29건(valid), AUC=0.8670
  fold 2: 양성 115건(train) / 29건(valid), AUC=0.9021
  fold 3: 양성 115건(train) / 29건(valid), AUC=0.8053
  fold 4: 양성 116건(train) / 28건(valid), AUC=0.8627
  fold 5: 양성 115건(train) / 29건(valid), AUC=0.8669
OOF AUROC: 0.8595, 부트스트랩 95% CI: [0.8297, 0.8867]


## 7. 3단 비교표 재현 (문서 검증용)

문서에 명시된 참고치(CB 완전정보 ≈0.99, A1 0.858, A2 0.820 — 전부 인구 포함 기준)와
이번 재현 결과를 나란히 놓아 재현성을 확인한다.

In [14]:
compare_table = pd.DataFrame([
    {'모델': 'CB 완전정보 (참고, 재현 대상 아님)', 'AUROC': 0.99, '비고': '기존 신용평가 상한'},
    {'모델': 'A1 (신청정보 포함, F0)', 'AUROC': round(auc_a1_24, 4), '비고': '문서 참고치 0.858'},
    {'모델': 'A2 F0 (인구 포함)', 'AUROC': round(results_a2_24m['F0 (인구 포함)']['oof_auc'], 4), '비고': '문서 참고치 0.820'},
    {'모델': 'A2 F1 (성별만 제외)', 'AUROC': round(results_a2_24m['F1 (성별만 제외)']['oof_auc'], 4), '비고': '문서 참고치 0.792'},
    {'모델': 'A2 F2 (공식, 인구 전부 제외)', 'AUROC': round(results_a2_24m['F2 (인구 전부 제외, 공식)']['oof_auc'], 4), '비고': '문서 참고치 0.679 [0.638, 0.721] ★공식수치'},
])
compare_table


,모델,AUROC,비고
0,"CB 완전정보 (참고, 재현 대상 아님)",0.9900,기존 신용평가 상한
1,"A1 (신청정보 포함, F0)",0.8595,문서 참고치 0.858
2,A2 F0 (인구 포함),0.8190,문서 참고치 0.820
3,A2 F1 (성별만 제외),0.7933,문서 참고치 0.792
4,"A2 F2 (공식, 인구 전부 제외)",0.6896,"문서 참고치 0.679 [0.638, 0.721] ★공식수치"


## 8. apply 스코어(SCORE_A2)와의 방향 일치성 점검

같은 대상은 아니지만(학습 68,677명 vs apply 794,773명 씬파일러), 새로 학습한 A2 F2 모델의
OOF 점수 분포와 기존 apply 점수(`SCORE_A2`) 분포가 비슷한 스케일·분포 형태를 보이는지
육안으로 대조해서 큰 이상이 없는지 확인한다(정식 검증이 아니라 sanity check).

In [15]:
df_apply = pd.read_csv(f'{handoff_path}/track_a_apply_v2_scored_R.csv')

print("기존 apply SCORE_A2 분포:")
print(df_apply['SCORE_A2'].describe())
print()
print("이번 재현 A2 F2 OOF 점수 분포 (TARGET_24M 기준):")
oof_proba_f2 = oof_arrays_a2_24m['F2 (인구 전부 제외, 공식)']  # 4절에서 저장해둔 값 재사용 (재계산 안 함)
print(pd.Series(oof_proba_f2).describe())

print("\n[주의] 두 점수 모두 확률이 아니라 다운샘플 학습으로 부풀려진 값 — 순위·분포 형태 비교로만 사용")


기존 apply SCORE_A2 분포:
count    794773.000000
mean          0.025416
std           0.032526
min           0.000552
25%           0.006722
50%           0.012135
75%           0.034299
max           0.537741
Name: SCORE_A2, dtype: float64

이번 재현 A2 F2 OOF 점수 분포 (TARGET_24M 기준):
count    68677.000000
mean         0.019137
std          0.023006
min          0.000228
25%          0.007933
50%          0.015214
75%          0.021246
max          0.714400
dtype: float64

[주의] 두 점수 모두 확률이 아니라 다운샘플 학습으로 부풀려진 값 — 순위·분포 형태 비교로만 사용


## 9. 결과 저장

In [16]:
import os
os.makedirs(handoff_path, exist_ok=True)

compare_table.to_csv(f'{handoff_path}/track_a_model_comparison_final.csv', index=False, encoding='utf-8-sig')

with open(f'{handoff_path}/track_a_model_results.json', 'w', encoding='utf-8') as f:
    json.dump({
        'A2_TARGET_24M': results_a2_24m,
        'A2_TARGET_12M': results_a2_12m,
        'A1_TARGET_24M_F0': {'oof_auc': auc_a1_24, 'bootstrap_ci': ci_range_a1},
    }, f, ensure_ascii=False, indent=2, default=str)

print("저장 완료: track_a_model_comparison_final.csv, track_a_model_results.json")


저장 완료: track_a_model_comparison_final.csv, track_a_model_results.json
